# Step 2: Cross-validated baseline and the input comparison
### Neeraj, MMAI 894 (router fine-tuning)

**Goal:** rebuild the original model (XLM-RoBERTa with a type head and an escalation head, same recipe as the first notebook) inside the new cross-validation harness, and answer three questions:

1. **Does the harness give a sane picture?** The original notebook found type accuracy in the 60s to 70s, swinging from run to run. We expect something similar, now measured on 160 chats (each scored by models that never saw it) instead of 40.
2. **How many epochs?** One epoch count, chosen once, from the average curve.
3. **Does the input matter?** We train on three different views of the chat and compare them on identical folds:
   - `first_message`: only the customer's first message (what a real router sees)
   - `customer_turns`: the customer's first and last messages (what the original notebook used)
   - `full_conversation`: customer, agent, customer (what the team's pipeline passes today)

**What you need to do:** switch the **GPU on** (Accelerator: GPU T4), Internet on, then run all cells. The long training cell can take an hour or more; if the session stops, just run that cell again, because finished runs are skipped. At the end, download `results_02.zip` and the figures listed in the last cell.

**How it is organised:** all logic lives in the repo (`protocol.py`, `model.py`, `train_cv.py`, `cv_analysis.py`, `metrics.py`); this notebook only runs it and explains the results. Every number shown is computed from saved files, never typed in.

## Step 0: Get the latest code
Same as notebook 01: delete any old copy, clone fresh, clear stale imports, and show which commit is running.

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

# Always start from a fresh copy of the latest code (Kaggle keeps files between runs in one session).
%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}          # download the repo (public, so no login is needed)

%cd {REPO_DIR}
sys.path.insert(0, "src")                    # lets us write: from northstar.finetune import protocol
importlib.invalidate_caches()                # forget what Python saw in the folder on an earlier run
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]                    # drop stale copies of our modules from earlier runs

assert os.path.exists("src/northstar/finetune/train_cv.py"), "train_cv.py is missing: has the latest code been pushed to GitHub?"
!git log --oneline -3                        # shows exactly which version of the code is running

## Step 1: Check the machine
Training needs a GPU. If the next cell says `cuda: False`, stop: change the notebook's Accelerator to GPU T4 and restart.

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "No GPU found. Set Accelerator to GPU T4 in the notebook settings."

import pandas as pd
from northstar.finetune import protocol as P
from northstar.finetune.train_cv import TrainConfig, run_cv

df = P.load_all()                 # refuses to run if the dataset file is not the exact team file
pool = P.pool(df)                 # the 160 chats; the 40 test chats are not in here
print("Pool:", len(pool), "chats | folds file:", P.FOLDS_PATH.name)

## Step 2: Settings
Everything you might want to change is in this one cell. The defaults reproduce the original notebook's recipe: XLM-RoBERTa base, learning rate 2e-5, batch size 16, 30 epochs, constant learning rate, no warmup, no gradient clipping. Two harmless differences make it faster: each batch is padded only to its longest chat, and the arithmetic runs in half precision on the GPU.

If the time estimate in Step 3 is too long for you, lower `SEEDS` (for example to `(11, 22)`) or `VARIANTS` (for example drop `full_conversation`), run, and add the rest later: finished runs are never repeated.

In [ ]:
OUT = "/kaggle/working/cv_old_recipe"        # where every run is saved (one small file per run)
RESULTS = "/kaggle/working/results_02"       # tables and figures to download
os.makedirs(RESULTS, exist_ok=True)

VARIANTS = P.INPUT_VARIANTS                  # ('first_message', 'customer_turns', 'full_conversation')
SEEDS = P.CV_SEEDS                           # (11, 22, 33): three different random starts per fold
EPOCHS = 30
cfg = TrainConfig(epochs=EPOCHS)             # all other settings are the original recipe (see train_cv.TrainConfig)
print(cfg)

## Step 3: Smoke test and time estimate
One tiny run (one fold, one seed, two epochs) to prove the whole pipeline works end to end before we commit an hour of GPU time, and to measure how long an epoch takes. The estimate is rough and slightly pessimistic, because it includes the one-off cost of loading the model.

In [ ]:
import time

t0 = time.time()
run_cv(cfg, "customer_turns", "/kaggle/working/smoke", folds=[0], seeds=[11], epochs=2)
seconds = time.time() - t0

n_runs = len(VARIANTS) * len(SEEDS) * P.N_FOLDS
estimate = seconds / 2 * EPOCHS * n_runs / 60
print(f"Smoke test: {seconds:.0f}s for 2 epochs. Full run = {n_runs} runs x {EPOCHS} epochs, "
      f"estimated at about {estimate:.0f} minutes.")

## Step 4: The full cross-validation
For every input variant, every seed and every fold: train a fresh model on four folds and score the fifth after each epoch, saving the raw scores. Nothing is selected here; we only collect honest held-out predictions.

**This is the long cell.** It prints one line per finished run. If it is interrupted, run it again: it skips what is already saved.

In [ ]:
for variant in VARIANTS:
    print(f"=== input variant: {variant}")
    run_cv(cfg, variant, OUT, seeds=SEEDS)
print("All runs finished.")

## Step 5: Put the folds back together
For each seed, the five folds' held-out scores are joined so that all 160 chats have a score from a model that never saw them ("out-of-fold"). The loader refuses an incomplete set, so a half-finished run cannot be analysed by mistake.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from northstar.finetune import cv_analysis as A
from northstar.finetune import metrics as M

runs, oof, curves = {}, {}, {}
for v in VARIANTS:
    runs[v] = A.load_variant(OUT, v)               # every saved run of this input variant
    oof[v] = A.assemble(runs[v], pool)             # every chat scored by models that never saw it
    curves[v] = A.epoch_curves(oof[v], pool)       # accuracy and friends, per seed and epoch
print({v: f"{len(runs[v])} runs, {len(oof[v])} seeds" for v in VARIANTS})

## Step 6: How many epochs?
An epoch is one pass over the training chats. Too few and the model has not learned; too many and it starts memorising. **Left plot:** held-out macro-F1 by epoch (mean over seeds, shaded band = 1 standard deviation across seeds). **Right plot:** training loss (dashed) against held-out loss (solid); when the held-out loss stops falling or rises while the training loss keeps falling, the model is memorising.

We choose **one** epoch count for everything: smooth the average curve over three neighbouring epochs (single epochs are noisy) and take its best point. Note that this choice is made on the same cross-validation it is then measured on, so the scores below are very slightly optimistic; the test chats stay untouched.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for v in VARIANTS:
    g = curves[v].groupby("epoch")["type_macro_f1"]
    mean, sd = g.mean(), g.std()
    axes[0].plot(mean.index, mean, label=v)
    axes[0].fill_between(mean.index, mean - sd, mean + sd, alpha=0.15)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("macro-F1 on held-out chats")
axes[0].set_title("Request type: held-out macro-F1 (mean over seeds, band = 1 sd)"); axes[0].legend()

for v in VARIANTS:
    lc = A.loss_curves(runs[v]).groupby("epoch")[["train_loss", "val_loss"]].mean()
    line, = axes[1].plot(lc.index, lc["val_loss"], label=f"{v} held-out")
    axes[1].plot(lc.index, lc["train_loss"], "--", color=line.get_color(), label=f"{v} train")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("loss"); axes[1].set_title("Loss: training vs held-out"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{RESULTS}/learning_curves.png", dpi=150); plt.show()

# One common epoch count: the best point of the smoothed average curve over all variants and seeds.
all_curves = pd.concat([curves[v].assign(variant=v) for v in VARIANTS])
E_STAR, smooth = A.pick_epoch(all_curves)
print("Chosen epoch count (E*):", E_STAR, "| smoothed macro-F1 there:", round(float(smooth.loc[E_STAR]), 3))

## Step 7: Results at the chosen epoch
One row per input variant, all on identical folds. The numbers are **mean ± standard deviation over the three seeds**: the standard deviation is the run-to-run luck we found in the original notebook. Escalation is scored at the natural threshold (positive score = escalate); `missed` counts chats that needed a person but were not escalated. The last columns average the three seeds' scores before deciding ("seed ensemble"), a cheap way to cut luck.

In [ ]:
n_esc = int(pool["escalation_label"].sum())
summ = {v: A.summary_at_epoch(oof[v], pool, E_STAR) for v in VARIANTS}

rows, numeric = [], []
for v in VARIANTS:
    ps = summ[v]["per_seed"]
    def ms(c): return f"{ps[c].mean():.3f} ± {ps[c].std():.3f}"
    et, ee = summ[v]["ensemble_type"], summ[v]["ensemble_escalation"]
    rows.append({"input": v, "type accuracy": ms("type_accuracy"), "type macro-F1": ms("type_macro_f1"),
                 "escalation recall": ms("esc_recall"), "escalation precision": ms("esc_precision"),
                 f"missed (of {n_esc})": f"{ps['esc_missed'].mean():.1f}", "ensemble type acc": f"{et['accuracy']:.3f}",
                 "ensemble macro-F1": f"{et['macro_f1']:.3f}", "ensemble esc recall": f"{ee['recall']:.3f}"})
    numeric.append(dict(input=v, epoch=E_STAR,
                        **{f"{c}_mean": ps[c].mean() for c in ps.columns if c != "seed"},
                        **{f"{c}_sd": ps[c].std() for c in ps.columns if c != "seed"},
                        ensemble_type_accuracy=et["accuracy"], ensemble_type_macro_f1=et["macro_f1"],
                        ensemble_esc_recall=ee["recall"], ensemble_esc_precision=ee["precision"], ensemble_esc_f2=ee["f2"]))
table = pd.DataFrame(rows).set_index("input")
pd.DataFrame(numeric).to_csv(f"{RESULTS}/cv_summary.csv", index=False)
table

## Step 8: Which input is primary?
The rule was written **before** seeing any results (`docs/EVAL_PROTOCOL.md`, section 4): the primary input is `first_message` if its macro-F1 is within one seed-to-seed standard deviation of the best input; otherwise the best input is primary, and the report states that the production-realistic input costs accuracy. All three stay in the report either way.

In [ ]:
means = {v: summ[v]["per_seed"]["type_macro_f1"].mean() for v in VARIANTS}
best = max(means, key=means.get)
sd_best = summ[best]["per_seed"]["type_macro_f1"].std()
gap = means[best] - means["first_message"]
primary = "first_message" if gap <= sd_best else best

print("Mean macro-F1 by input:", {v: round(float(m), 3) for v, m in means.items()})
print(f"Best input: {best} | its seed-to-seed sd: {sd_best:.3f} | first_message is {gap:.3f} behind the best")
print("=> Primary input by the rule:", primary)
if primary != "first_message":
    print("   Report note: the production-realistic input (first message only) costs accuracy.")

## Step 9: How sure can we be?
**Intervals:** with only 160 chats, a single accuracy number hides a lot of uncertainty. We resample *customers* (not single chats, because one customer's chats are not independent) 2,000 times and show the middle 95% of the resulting accuracies, for the seed ensemble.

**Comparisons:** for two inputs scored on the same chats, only the chats where exactly one input is right matter. The exact McNemar test asks whether that split is lopsided enough not to be luck. We list *every* pair, not only favourable ones.

In [ ]:
import itertools

y_type = pool["type_label"].values
customers = pool["customer_id"].values
ens_pred = {v: summ[v]["ensemble_type_logits"].argmax(-1) for v in VARIANTS}
correct = {v: ens_pred[v] == y_type for v in VARIANTS}

ci_rows = []
for v in VARIANTS:
    lo, hi = M.cluster_bootstrap(lambda idx, c=correct[v]: c[idx].mean(), customers, n_boot=2000)
    ci_rows.append(dict(input=v, ensemble_accuracy=correct[v].mean(), ci_low=lo, ci_high=hi))
ci = pd.DataFrame(ci_rows).set_index("input")

pair_rows = []
for a, b in itertools.combinations(VARIANTS, 2):
    r = M.mcnemar_exact(correct[a], correct[b])
    pair_rows.append(dict(a=a, b=b, only_a_right=r["only_a_right"], only_b_right=r["only_b_right"], p_value=r["p_value"]))
pairs = pd.DataFrame(pair_rows)

ci.round(3).to_csv(f"{RESULTS}/accuracy_intervals.csv"); pairs.to_csv(f"{RESULTS}/input_comparisons.csv", index=False)
print("Accuracy of the seed ensemble with 95% interval (resampling customers):")
display(ci.round(3))
print("All pairwise comparisons (exact McNemar; small p means a real difference, not luck):")
display(pairs.round(3))

## Step 10: Where does it go wrong?
For the primary input (seed ensemble): which request types are confused with which, how many chats of each type we get right, and whether English or French is harder. Rows of the matrix are the true type, columns what the model said; anything off the diagonal is a mistake.

In [ ]:
from sklearn.metrics import confusion_matrix

pred = ens_pred[primary]
labels = list(range(len(P.REQUEST_TYPES)))
cm = confusion_matrix(y_type, pred, labels=labels)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(labels); ax.set_xticklabels(P.REQUEST_TYPES, rotation=90)
ax.set_yticks(labels); ax.set_yticklabels(P.REQUEST_TYPES)
for i in labels:
    for j in labels:
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xlabel("model said"); ax.set_ylabel("true type"); ax.set_title(f"Out-of-fold confusion, input = {primary}")
plt.tight_layout(); plt.savefig(f"{RESULTS}/confusion_{primary}.png", dpi=150); plt.show()

per_type = pd.DataFrame({"chats": cm.sum(1), "recall": M.per_class_recall(y_type, pred, len(labels))}, index=P.REQUEST_TYPES)
by_language = pd.Series(pred == y_type).groupby(pool["language"].values).agg(["mean", "count"]).rename(columns={"mean": "accuracy", "count": "chats"})
per_type.round(2).to_csv(f"{RESULTS}/per_type_recall_{primary}.csv"); by_language.round(3).to_csv(f"{RESULTS}/by_language_{primary}.csv")
display(per_type.round(2)); display(by_language.round(3))

## Step 11: Does this match the original notebook?
A plausibility check, not an exact match: the original notebook scored **40 test chats** five times and found a wide range. We score **160 held-out chats** (out-of-fold), so the numbers should be in the same neighbourhood but need not agree. The original numbers are read from `results/stability_runs.csv` in the repo.

In [ ]:
old = pd.read_csv("src/northstar/finetune/results/stability_runs.csv")
new = summ["customer_turns"]["per_seed"]       # customer_turns is the input the original notebook used

print(f"Original notebook (5 reruns, 40 test chats): type accuracy mean {old['type_accuracy'].mean():.3f}, "
      f"range {old['type_accuracy'].min():.3f} to {old['type_accuracy'].max():.3f}")
print(f"This harness (3 seeds, 160 out-of-fold chats, epoch {E_STAR}): type accuracy mean {new['type_accuracy'].mean():.3f}, "
      f"range {new['type_accuracy'].min():.3f} to {new['type_accuracy'].max():.3f}")

## Step 12: Save the results
The zip holds the raw scores of every run, so thresholds, calibration, seed ensembles and the cascade can be studied later without retraining. Download these files from the Output tab (`results_02` folder), put the tables and figures into `results/step02/` and the unzipped runs into `results/cv_old_recipe/` in the repo, and tell me.

In [ ]:
shutil.make_archive(f"{RESULTS}/cv_old_recipe", "zip", OUT)       # all saved runs, a few MB
for name in sorted(os.listdir(RESULTS)):
    print(f"{name:45s} {os.path.getsize(os.path.join(RESULTS, name)) / 1024:8.0f} KB")

## What this notebook shows, and its limits
- **Shows:** a cross-validated baseline for the original recipe, one chosen epoch count, and a like-for-like comparison of the three inputs with intervals and every pairwise test. Read the tables and plots above; this text deliberately holds no numbers.
- **Decisions it feeds:** the epoch count and the primary input for every later step; whether a production-realistic input (first message only) is good enough.
- **Limits:**
  - The epoch count is chosen on the same folds it is scored on, so scores are very slightly optimistic.
  - Three seeds measure run-to-run luck only roughly.
  - This is still the original recipe (constant learning rate, no warmup, no gradient clipping) and only two heads (type, escalation); the recipe, LoRA, language and reason heads come in later steps.
  - One generator wrote all the chats, and the middle line of `full_conversation` is an AI-written agent reply that may hint at the answer, so its score is probably flattering.
  - Cross-validation numbers on 160 chats and test numbers on 40 chats are not directly comparable.